# TOQIX AI — 03 · Universal job runner (website → GPU → finished media)

Run this notebook on a **GPU runtime** (Runtime → Change runtime type → **GPU**;
a free T4 is enough), then press **Run All** and leave it open. It connects
TOQIX to its generation engines:

1. Every 30 seconds it asks the TOQIX website for waiting jobs
   (`GET https://toqix.pythonanywhere.com/api/jobs` — jobs with status
   `queued`, plus the older `exported` status, count as waiting).
2. It **atomically claims** the **oldest** waiting job (see the worker
   mesh below — only one worker can win a claim), reports **live
   progress** back to the website while it works, generates the media,
   and **uploads the finished file** to the job — the website then
   shows it as done in the gallery.

| Job kind | Engine (all open weights, no external API) | Output |
|---|---|---|
| video | Wan 2.1 T2V 1.3B (proven T4 recipe from notebook 01) | .mp4, h264 / yuv420p / faststart |
| image | SDXL-Turbo (4 steps) | .jpg |
| audio | Kokoro-82M text-to-speech (character voices) | .mp3 |

**About video speed — checked honestly (2026-10-14):** we verified the
LightX2V Hugging Face org looking for a step-distillation LoRA for
Wan2.1-T2V-**1.3B**. Their 4-step LoRAs exist only for the **14B** models
(`lightx2v/Wan2.1-Distill-Loras` — `wan2.1_t2v_14b_lora_rank64_lightx2v_4step`),
which do not fit a free T4. For the 1.3B there is no diffusers-loadable
step-distillation LoRA: `lightx2v/Wan2.1-T2V-1.3B-Distill-Models` is a single
raw `.pt` checkpoint for the LightX2V framework (not a LoRA), and the 1.3B
"longcat" repos are GRPO quality LoRAs that still need normal step counts.
So instead of inventing a repo id, video runs the base model at **14 steps**
instead of the old 30 — about **2× faster** with the exact memory recipe
that already produced TOQIX's first video on a T4. The runner always renders
video with the 1.3B model, even if a job record names the 14B (the 14B does
not fit this GPU).

Models load **lazily** — a model is only downloaded/loaded when the first
job of its kind arrives — and stay resident for later jobs. If the GPU runs
out of memory, the runner frees the other models and retries the job once.
Any job that still fails is marked **failed** on the website and the runner
moves on to the next job. To stop the runner, interrupt the last cell.

**Fast lane (2026-10-14):** video jobs now denoise small (640x368, 8 fps) and finish big in ffmpeg (motion interpolation to 16 fps + upscale to the job size) — see the video cell. First run must be timed end-to-end before any speed claim.


## One queue, many GPUs — the worker mesh

This notebook is a **worker**. Run the **same notebook** on any free
GPU — a Colab session on any Google account, a Kaggle notebook
(with **Internet ON**), a local machine — and every copy joins the
same workforce, all pulling from the one website queue:

- Each worker has a name (watch the *Worker identity* cell, e.g.
  `colab-a3f9`, `kaggle-m71q`) and takes **one job at a time**.
- Claiming is **atomic**: two workers can **never** take the same
  job. The worker that loses a claim simply moves on to the next
  waiting job.
- While a worker generates, its progress posts double as a
  **heartbeat**. If a free session dies silently, the website
  notices the missing heartbeat after **45 minutes** and returns
  the job to the queue for another worker — nothing gets stuck.
- Because the load is spread across sessions and accounts, no
  single free GPU quota burns out fast.


In [ ]:
import subprocess
import torch

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import subprocess
import sys


def _pip_install(packages, label):
    """Install packages, preferring prebuilt wheels. Never raises: one
    bad package must not kill the whole worker setup."""
    print("installing", label, ":", " ".join(packages), flush=True)
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--prefer-binary",
         *packages])
    ok = result.returncode == 0
    print(label, "install", "OK" if ok else "FAILED (see output above)",
          flush=True)
    return ok


# Core packages -- all publish prebuilt wheels for Colab's Python.
_pip_install(
    ["diffusers>=0.33.0", "transformers", "accelerate", "bitsandbytes",
     "protobuf", "imageio", "imageio-ffmpeg", "soundfile", "requests"],
    "core")

# Risky packages -- installed one by one so a failure is visible and
# non-fatal (a dead install here only disables that feature; the worker
# still starts). sentencepiece is pinned to 0.2.2, the first release
# shipping prebuilt CPython 3.13 wheels: unpinned installs on Colab's
# Python 3.13 image picked a source-only release, compiled C++ for
# ~15 minutes and failed, and the old check=True aborted the whole
# setup before the worker cell ever ran (fleet attempt 1, 2026-10-14).
_pip_install(["sentencepiece==0.2.2"], "sentencepiece")
_pip_install(["kokoro"], "kokoro")

print("setup finished (a FAILED line above only disables that feature)")

import gc
import os
import time
import traceback

import requests
import torch

SITE = "https://toqix.pythonanywhere.com"
WORK_DIR = "/content/toqix_runner"
os.makedirs(WORK_DIR, exist_ok=True)

# ffmpeg: prefer the binary bundled with imageio-ffmpeg (always present in
# Colab after the install above); fall back to a system ffmpeg.
try:
    import imageio_ffmpeg
    FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
except Exception:
    FFMPEG = "ffmpeg"
print("site:", SITE)
print("ffmpeg:", FFMPEG)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def get_jobs():
    """Return the website's job list (list of dicts)."""
    resp = requests.get(SITE + "/api/jobs", timeout=30)
    resp.raise_for_status()
    return resp.json()


def set_progress(job_id, progress, status):
    """Report progress/status for a job. Never raises — a website hiccup
    must not kill a generation that is already running.

    Once the Worker identity cell has run, every post also carries this
    worker's name — the server treats each post as a claim HEARTBEAT
    and refreshes the job's claim stamp, so a live worker's job is
    never mistaken for a dead worker's job."""
    payload = {"progress": int(progress), "status": status}
    worker = globals().get("WORKER")
    if worker:
        payload["worker"] = worker
    try:
        requests.post(
            SITE + "/api/jobs/%s/progress" % job_id,
            json=payload,
            timeout=30,
        )
    except Exception as exc:
        print("progress post failed (continuing):", repr(exc))


_MIME = {".mp4": "video/mp4", ".jpg": "image/jpeg", ".jpeg": "image/jpeg",
         ".mp3": "audio/mpeg"}


def import_file(job_id, path):
    """Upload the finished media file. The server normalizes it and marks
    the job done. Returns True on success."""
    ext = os.path.splitext(path)[1].lower()
    mime = _MIME.get(ext, "application/octet-stream")
    with open(path, "rb") as fh:
        resp = requests.post(
            SITE + "/api/jobs/%s/import" % job_id,
            files={"file": (os.path.basename(path), fh, mime)},
            timeout=600,
        )
    print("import", job_id, "-> HTTP", resp.status_code, resp.text[:200])
    return resp.ok


def is_cuda_oom(exc):
    return isinstance(exc, torch.cuda.OutOfMemoryError) or (
        isinstance(exc, RuntimeError) and "out of memory" in str(exc).lower())


def free_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# ------------------------------------------------------------------
# Worker identity — this notebook is ONE worker in the TOQIX mesh.
#
# Run the SAME notebook on any free GPU (Colab on any account, Kaggle
# with Internet ON, a local machine) and every copy works the SAME
# website queue in parallel. A worker takes ONE job at a time, and
# the website's atomic claim means two workers can never grab the
# same job. If this session dies mid-job, the claim goes stale after
# 45 minutes without a progress heartbeat and the job returns to
# the queue for another worker.
# ------------------------------------------------------------------
import random
import string


def _detect_platform():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    if os.environ.get("COLAB_GPU") or os.environ.get("COLAB_TPU_ADDR"):
        return "colab"
    try:
        import google.colab  # noqa: F401
        return "colab"
    except Exception:
        pass
    return "local"


_suffix = "".join(random.choices(string.ascii_lowercase + string.digits, k=4))
WORKER = "%s-%s" % (_detect_platform(), _suffix)
print("worker identity:", WORKER)


def claim_job(job_id):
    """Atomically claim a waiting job for THIS worker.

    Returns (True, job_dict) when this worker won the claim, and
    (False, None) when another worker already holds the job (the
    website answers 409) or the claim could not be posted. Never
    raises — the loop simply moves on to the next waiting job."""
    try:
        resp = requests.post(
            SITE + "/api/jobs/%s/claim" % job_id,
            json={"worker": WORKER},
            timeout=30,
        )
        if resp.status_code == 200:
            return True, resp.json()
        if resp.status_code == 409:
            return False, None
        print("claim %s -> HTTP %s (treated as not claimable)"
              % (job_id, resp.status_code))
        return False, None
    except Exception as exc:
        print("claim post failed (continuing):", repr(exc))
        return False, None


## Video engine — Wan 2.1 T2V 1.3B (proven T4 recipe + 14-step speed layer)

Same memory recipe that generated TOQIX's first video on a free T4
(notebook 01): the UMT5-XXL text encoder loads in **8-bit** (bitsandbytes)
pinned to the GPU — a bf16 encoder is ~11 GB and OOM-crashes the ~12.7 GB
free-Colab system RAM before offload even starts. Each job's prompt is
encoded **once**, then the encoder is stripped from the pipeline before
denoising (it is re-loaded from the local cache for the next job — a small
cost that keeps the proven memory profile for every job). Denoising runs
with **sequential CPU offload**, transformer + embeddings in **float16**
(the T4/Turing has no bf16 attention kernels — efficient SDPA accepts
Half/Float only) and the native-efficient attention backend (the math path
materializes an ~18 GiB attention matrix and VRAM-OOMs the T4). The VAE
stays fp32 with tiling + slicing.

Speed layer: **14 inference steps, guidance 5.0** (see the intro note for
why no distilled LoRA is used), live per-step progress (5% → 95%), frame
count from the job's duration at 16 fps (capped at 121 frames), size from
the job's aspect ratio, and a final ffmpeg pass to h264 / yuv420p /
limited-range BT.709 / faststart.


In [ ]:
from diffusers import AutoencoderKLWan, WanPipeline, WanTransformer3DModel
from diffusers.utils import export_to_video
from transformers import BitsAndBytesConfig, UMT5EncoderModel

VIDEO_MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
VIDEO_STEPS = 14          # speed layer: 14 steps instead of the old 30
VIDEO_GUIDANCE = 5.0
VIDEO_FPS = 16
VIDEO_SIZES = {"16:9": (832, 480), "9:16": (480, 832), "1:1": (640, 640)}
# FAST LANE (generate-small-finish-big, added 2026-10-14): denoise at a
# smaller size and HALF the frame rate, then let ffmpeg finish big --
# motion interpolation back to 16 fps + lanczos upscale to the job size.
# Denoising cost scales ~ frames x pixels, so this cuts it to roughly
# 0.5 x 0.58 = ~0.29x of the full-size 14-step run (~3.4x faster).
# Set FAST_VIDEO = False to return to full-size generation.
FAST_VIDEO = True
FAST_SIZES = {"16:9": (640, 368), "9:16": (368, 640), "1:1": (512, 512)}
FAST_GEN_FPS = 8          # frames generated at 8 fps, finished to 16
DEFAULT_NEGATIVE = (
    "low quality, worst quality, blurry, distorted, deformed, extra limbs, "
    "flickering, jittery motion, static frame, watermark, text, subtitles")

_video = {"pipe": None, "offload_ready": False}


def _load_video_encoder():
    """Fresh 8-bit UMT5 encoder, pinned to the GPU (~5.7 GB, not ~11 GB)."""
    return UMT5EncoderModel.from_pretrained(
        VIDEO_MODEL_ID, subfolder="text_encoder",
        quantization_config=BitsAndBytesConfig(load_in_8bit=True),
        device_map={"": "cuda"}, torch_dtype=torch.bfloat16)


def ensure_video_pipe():
    if _video["pipe"] is not None:
        return _video["pipe"]
    print("loading Wan 2.1 video pipeline (first video job)...")
    transformer = WanTransformer3DModel.from_pretrained(
        VIDEO_MODEL_ID, subfolder="transformer", torch_dtype=torch.float16)
    vae = AutoencoderKLWan.from_pretrained(
        VIDEO_MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
    encoder = _load_video_encoder()
    pipe = WanPipeline.from_pretrained(
        VIDEO_MODEL_ID, text_encoder=encoder, transformer=transformer,
        vae=vae, torch_dtype=torch.float16)
    try:
        pipe.vae.enable_tiling()
        pipe.vae.enable_slicing()
    except Exception:
        pass
    _video["pipe"] = pipe
    print("video pipeline ready")
    return pipe


def free_video():
    if _video["pipe"] is not None:
        del _video["pipe"]
        _video["pipe"] = None
        _video["offload_ready"] = False
    free_cuda()


def generate_video(job):
    """Generate one video job; returns the final .mp4 path."""
    jid = job["id"]
    pipe = ensure_video_pipe()

    duration = int(round(float(job.get("duration") or 5)))
    target_w, target_h = VIDEO_SIZES.get(job.get("aspect") or "16:9",
                                         VIDEO_SIZES["16:9"])
    if FAST_VIDEO:
        gen_fps = FAST_GEN_FPS
        num_frames = min(duration * gen_fps + 1, 121)
        width, height = FAST_SIZES.get(job.get("aspect") or "16:9",
                                       FAST_SIZES["16:9"])
        print("FAST LANE: generating %d frames at %dx%d / %d fps, "
              "finishing to %dx%d / %d fps"
              % (num_frames, width, height, gen_fps,
                 target_w, target_h, VIDEO_FPS))
    else:
        gen_fps = VIDEO_FPS
        num_frames = min(duration * VIDEO_FPS + 1, 121)
        width, height = target_w, target_h
    try:
        seed = int(job.get("seed", 42))
    except Exception:
        seed = 42
    prompt = job.get("enhanced_prompt") or job.get("prompt") or ""
    negative = job.get("negative_prompt") or DEFAULT_NEGATIVE

    # Encode this job's prompt ONCE with the GPU-resident 8-bit encoder,
    # then strip the encoder from the pipeline before denoising — the
    # single biggest RAM resident, and denoising only needs the embeddings.
    encoder = pipe.text_encoder
    if encoder is None:
        encoder = _load_video_encoder()
        pipe.text_encoder = encoder
    prompt_embeds, negative_prompt_embeds = pipe.encode_prompt(
        prompt=prompt, negative_prompt=negative,
        do_classifier_free_guidance=True, device="cuda")
    pipe.text_encoder = None
    del encoder
    gc.collect()
    torch.cuda.empty_cache()

    # Offload + attention backend are set up AFTER the strip (once), so the
    # offload hooks never re-stage the encoder onto the CPU.
    if not _video["offload_ready"]:
        try:
            pipe.enable_sequential_cpu_offload()
        except Exception:
            pipe.enable_model_cpu_offload()
        try:
            pipe.transformer.set_attention_backend("_native_efficient")
        except Exception:
            pass
        _video["offload_ready"] = True

    def on_step_end(pipe_, step_index, timestep, callback_kwargs):
        pct = 5 + int(90 * (step_index + 1) / VIDEO_STEPS)
        set_progress(jid, min(pct, 95), "generating")
        return callback_kwargs

    generator = torch.Generator(device="cpu").manual_seed(seed)
    frames = pipe(
        prompt_embeds=prompt_embeds,
        negative_prompt_embeds=negative_prompt_embeds,
        height=height, width=width, num_frames=num_frames,
        guidance_scale=VIDEO_GUIDANCE, num_inference_steps=VIDEO_STEPS,
        generator=generator, callback_on_step_end=on_step_end,
    ).frames[0]

    raw_path = os.path.join(WORK_DIR, jid + "_raw.mp4")
    export_to_video(frames, raw_path, fps=gen_fps)

    # Delivery encode: h264 / yuv420p / limited-range BT.709 / faststart.
    final_path = os.path.join(WORK_DIR, jid + ".mp4")
    cmd = [FFMPEG, "-y", "-i", raw_path,
           "-c:v", "libx264", "-pix_fmt", "yuv420p",
           "-colorspace", "bt709", "-color_primaries", "bt709",
           "-color_trc", "bt709", "-color_range", "tv",
           "-movflags", "+faststart", final_path]
    if FAST_VIDEO:
        # Finish big: interpolate 8 -> 16 fps and upscale to the job size
        # BEFORE the encoder settings (cmd[4] is right after the input).
        cmd[4:4] = ["-vf",
                    "minterpolate=fps=%d:mi_mode=mci:mc_mode=aobmc:"
                    "me_mode=bidir,scale=%d:%d:flags=lanczos"
                    % (VIDEO_FPS, target_w, target_h)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0 or not os.path.exists(final_path):
        print("ffmpeg re-encode failed, importing the raw export instead:",
              result.stderr[-500:] if result.stderr else "")
        return raw_path
    return final_path


## Image engine — SDXL-Turbo (4 steps)

`stabilityai/sdxl-turbo` in float16: 4 inference steps, guidance 0.0 (that
is how Turbo is designed to run). The picture is saved as PNG, converted to
an RGB **JPG** (quality 90), and the JPG is what gets imported.


In [ ]:
from diffusers import AutoPipelineForText2Image
from PIL import Image

IMAGE_MODEL_ID = "stabilityai/sdxl-turbo"
IMAGE_SIZES = {"1:1": (1024, 1024), "16:9": (1216, 704), "9:16": (704, 1216)}

_image = {"pipe": None}


def ensure_image_pipe():
    if _image["pipe"] is not None:
        return _image["pipe"]
    print("loading SDXL-Turbo image pipeline (first image job)...")
    pipe = AutoPipelineForText2Image.from_pretrained(
        IMAGE_MODEL_ID, torch_dtype=torch.float16, variant="fp16")
    pipe.to("cuda")
    _image["pipe"] = pipe
    print("image pipeline ready")
    return pipe


def free_image():
    if _image["pipe"] is not None:
        del _image["pipe"]
        _image["pipe"] = None
    free_cuda()


def generate_image(job):
    """Generate one image job; returns the final .jpg path."""
    jid = job["id"]
    pipe = ensure_image_pipe()
    width, height = IMAGE_SIZES.get(job.get("aspect") or "1:1",
                                    IMAGE_SIZES["1:1"])
    try:
        seed = int(job.get("seed", 42))
    except Exception:
        seed = 42
    prompt = job.get("enhanced_prompt") or job.get("prompt") or ""

    set_progress(jid, 10, "generating")
    generator = torch.Generator(device="cuda").manual_seed(seed)
    image = pipe(prompt=prompt, width=width, height=height,
                 num_inference_steps=4, guidance_scale=0.0,
                 generator=generator).images[0]
    set_progress(jid, 60, "generating")

    png_path = os.path.join(WORK_DIR, jid + ".png")
    jpg_path = os.path.join(WORK_DIR, jid + ".jpg")
    image.save(png_path)
    Image.open(png_path).convert("RGB").save(jpg_path, quality=90)
    set_progress(jid, 95, "generating")
    return jpg_path


## Audio engine — Kokoro-82M text-to-speech (character voices)

The job's **style** names the character; the runner maps it to a Kokoro
voice and speaks the job's **original prompt text** (`job["prompt"]`).
British voices (Emma, George) use Kokoro's British pipeline (`lang_code
"b"`), the others the American one (`"a"`). If the chosen voice fails to
load, it falls back to `af_heart`. Chunks are joined into one WAV
(soundfile) and converted to MP3 for import.


In [ ]:
import numpy as np
import soundfile as sf
from kokoro import KPipeline

KOKORO_SR = 24000
# Character name (job["style"]) -> Kokoro voice id.
VOICE_MAP = {
    "ava": "af_heart",
    "emma": "bf_emma",
    "michael": "am_michael",
    "george": "bm_george",
    "sky": "af_sky",
    "adam": "am_adam",
}
DEFAULT_VOICE = "af_heart"

_audio = {}  # lang_code -> KPipeline


def ensure_audio_pipe(lang_code):
    if lang_code not in _audio:
        print("loading Kokoro TTS pipeline (lang %s)..." % lang_code)
        _audio[lang_code] = KPipeline(lang_code=lang_code)
    return _audio[lang_code]


def free_audio():
    _audio.clear()
    free_cuda()


def _speak(text, voice):
    """Synthesize text with one voice; returns a float32 numpy array."""
    lang_code = "b" if voice.startswith("b") else "a"
    pipeline = ensure_audio_pipe(lang_code)
    chunks = []
    for _graphemes, _phonemes, audio in pipeline(text, voice=voice,
                                                  speed=1.0):
        if audio is not None:
            chunks.append(np.asarray(audio, dtype=np.float32))
    if not chunks:
        raise RuntimeError("Kokoro produced no audio")
    return np.concatenate(chunks)


def generate_audio(job):
    """Generate one audio (TTS) job; returns the final .mp3 path."""
    jid = job["id"]
    text = (job.get("prompt") or "").strip()
    if not text:
        raise ValueError("audio job has no text to speak")
    voice = VOICE_MAP.get(str(job.get("style") or "").strip().lower(),
                          DEFAULT_VOICE)

    set_progress(jid, 10, "generating")
    try:
        samples = _speak(text, voice)
    except Exception as exc:
        print("voice %s failed (%r) — falling back to %s"
              % (voice, exc, DEFAULT_VOICE))
        samples = _speak(text, DEFAULT_VOICE)
    set_progress(jid, 60, "generating")

    wav_path = os.path.join(WORK_DIR, jid + ".wav")
    mp3_path = os.path.join(WORK_DIR, jid + ".mp3")
    sf.write(wav_path, samples, KOKORO_SR)
    cmd = [FFMPEG, "-y", "-i", wav_path, "-codec:a", "libmp3lame",
           "-q:a", "2", mp3_path]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0 or not os.path.exists(mp3_path):
        print("ffmpeg mp3 encode failed, writing mp3 via soundfile:",
              result.stderr[-300:] if result.stderr else "")
        sf.write(mp3_path, samples, KOKORO_SR, format="MP3")
    set_progress(jid, 95, "generating")
    return mp3_path


## Run the runner

Leave this cell running. It polls the website every 20 seconds, takes the
oldest waiting job, generates it with the matching engine, and imports the
finished file (importing is what marks the job **done** on the website).
On any error the job is marked **failed** and the loop keeps going; on a
CUDA out-of-memory it first frees the other models and retries once.
Interrupt the cell (Runtime → Interrupt execution) to stop.


In [ ]:
POLL_SECONDS = 30
_handled_keys = set()  # dedupe keys handled in this runner session
WAITING_STATUSES = ("queued", "exported")  # "exported" = legacy waiting


def run_job(job):
    kind = job.get("kind") or "video"
    if kind == "image":
        return generate_image(job)
    if kind == "audio":
        return generate_audio(job)
    return generate_video(job)


def free_other_models(kind):
    if kind != "video":
        free_video()
    if kind != "image":
        free_image()
    if kind != "audio":
        free_audio()


def run_job_with_oom_retry(job):
    try:
        return run_job(job)
    except Exception as exc:
        if not is_cuda_oom(exc):
            raise
        print("CUDA out of memory — freeing the other models, retry once")
        free_other_models(job.get("kind") or "video")
        free_cuda()
        return run_job(job)


def claim_next(candidates):
    """Atomically claim one candidate job (oldest first) for THIS worker.

    Returns (dedupe_key, claimed_job) for the first claim that wins, or
    None when every candidate is already held by another worker — two
    workers can never generate the same job."""
    for key, cand in candidates:
        ok, claimed = claim_job(cand["id"])
        if ok:
            return key, claimed
        print("[%s] job %s is held by another worker — trying the next"
              % (WORKER, cand["id"]))
    return None


print("TOQIX worker %s live — polling %s every %ds. Interrupt to stop."
      % (WORKER, SITE, POLL_SECONDS))
while True:
    try:
        jobs = get_jobs()
        waiting = [j for j in jobs
                   if j.get("status") in WAITING_STATUSES]
        if not waiting:
            print(time.strftime("%H:%M:%S"), "no waiting jobs — sleeping")
        else:
            # Priority: real user submissions ("queued") before legacy
            # "exported" packages, oldest first within each group.
            waiting.sort(key=lambda j: (0 if j.get("status") == "queued"
                                        else 1,
                                        j.get("created_at") or 0))
            # In-session dedupe: identical kind+prompt+aspect+duration+
            # style jobs (the site queue currently holds several test
            # duplicates) are generated ONCE per runner session; the
            # copies stay untouched on the site instead of burning
            # several full generations each.
            fresh = []
            seen_keys = set()
            for cand in waiting:
                key = (cand.get("kind") or "video",
                       (cand.get("prompt") or "").strip().lower(),
                       cand.get("aspect"), cand.get("duration"),
                       cand.get("style"))
                if key in _handled_keys:
                    print("skipping duplicate this session:", cand["id"],
                          repr((cand.get("prompt") or "")[:50]))
                    continue
                if key in seen_keys:
                    continue  # same content already offered this round
                seen_keys.add(key)
                fresh.append((key, cand))
            if not fresh:
                print(time.strftime("%H:%M:%S"),
                      "only duplicates waiting — sleeping")
                time.sleep(POLL_SECONDS)
                continue
            won = claim_next(fresh)
            if won is None:
                print(time.strftime("%H:%M:%S"),
                      "every waiting job is held by another worker — sleeping")
                time.sleep(POLL_SECONDS)
                continue
            _key, job = won
            _handled_keys.add(_key)
            jid = job["id"]
            kind = job.get("kind") or "video"
            print("=" * 60)
            print("[%s] claimed job %s (kind=%s): %s"
                  % (WORKER, jid, kind, (job.get("prompt") or "")[:80]))
            started = time.time()
            try:
                out_path = run_job_with_oom_retry(job)
                if import_file(jid, out_path):
                    print("[%s] job %s done in %.0fs -> %s"
                          % (WORKER, jid, time.time() - started, out_path))
                else:
                    print("[%s] job %s: import failed" % (WORKER, jid))
                    set_progress(jid, 0, "failed")
            except Exception as exc:
                print("[%s] job %s failed: %r" % (WORKER, jid, exc))
                traceback.print_exc()
                set_progress(jid, 0, "failed")
    except Exception as exc:
        print("poll error (will retry):", repr(exc))
    time.sleep(POLL_SECONDS)
